# mlreport Full Classification Demo

In [ ]:
!rm -rf /content/mlreport
!git clone https://github.com/Lucas-Summers/mlreport.git
!pip install /content/mlreport

Cloning into 'mlreport'...
remote: Enumerating objects: 249, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 249 (delta 119), reused 206 (delta 80), pack-reused 0 (from 0)
Receiving objects: 100% (249/249), 4.71 MiB | 26.64 MiB/s, done.
Resolving deltas: 100% (119/119), done.
Processing ./mlreport
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mlreport: filename=mlreport-0.0.1-py3-none-any.whl size=50964 sha256=a302eb20285ff03f9377cc3cba29bdcaa44d739bc44f3cc6cc191991288f07ab
  Stored in directory: /tmp/pip-ephem-wheel-cache-gqr0p9lk/wheels/f3/fa/1e/a640ba52482c3b11728e91f1a9f484f0e1d50affdff61c1ac0
Successfully built mlreport
  Attempting uninstall: mlreport
    Found existing installation: mlreport 0.0.1
    Uninstalling mlreport-0.0.1:
      Successfully uninstalled mlreport-0.0.1


## Setup Data

In [ ]:
from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from mlreport import ComparisonReport, Report

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
author = "Lucas Summers"
theme = "light"

## Appendix A: Classification Train/Test Split

In [ ]:
split_model = GaussianNB()

split_model.fit(X_train, y_train)
y_pred_train = split_model.predict(X_train)
y_pred_test = split_model.predict(X_test)

split_report = Report(
    split_model,
    title="Appendix A: Iris Train/Test Split Report",
    author=author,
    description="Gaussian Naive Bayes evaluated on a standard train/test split.",
    theme=theme,
)

split_report.add_split("train", X_train, y_train, y_pred_train)
split_report.add_split("test", X_test, y_test, y_pred_test)
split_report.build().to_pdf("reports/iris_split_report.pdf").to_txt()

Iris Classification Report - Train/Test Split
Lucas Summers
Gaussian Naive Bayes evaluated on a standard train/test split.

Model Overview
+-----------------+----------------+
| Property        | Value          |
+=================+================+
| Name            | GaussianNB     |
+-----------------+----------------+
| Type            | Classification |
+-----------------+----------------+
| Sklearn         | True (v1.6.1)  |
+-----------------+----------------+
| Parameter Count | 2              |
+-----------------+----------------+

Dataset Overview
+--------------------+-------------+
| Property           | Value       |
+====================+=============+
| Features           | 4           |
+--------------------+-------------+
| Total Observations | 150         |
+--------------------+-------------+
| CV Folds           | None        |
+--------------------+-------------+
| Train              | 112 (74.7%) |
+--------------------+-------------+
| Test               | 38 (25

## Appendix B: Classification Cross-Validation

In [ ]:
cv_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=2.0, gamma="scale")),
    ]
)

cv_model.fit(X_train, y_train)

cv_report = Report(
    cv_model,
    title="Appendix B: Iris Cross-Validation Report",
    author=author,
    description="Support Vector Classifier evaluated with 5-fold stratified cross-validation.",
    theme=theme,
)

cv_report.add_crossval(X_train, y_train, cv=cv)
cv_report.build().to_pdf("reports/iris_cv_report.pdf").to_txt()

Iris Classification Report - Cross-Validation
Lucas Summers
Support Vector Classifier evaluated with 5-fold stratified cross-validation.

Model Overview
+-----------------+----------------+
| Property        | Value          |
+=================+================+
| Name            | SVC            |
+-----------------+----------------+
| Type            | Classification |
+-----------------+----------------+
| Sklearn         | True (v1.6.1)  |
+-----------------+----------------+
| Parameter Count | 23             |
+-----------------+----------------+

Dataset Overview
+--------------------+---------+
| Property           |   Value |
+====================+=========+
| Features           |       4 |
+--------------------+---------+
| Total Observations |     112 |
+--------------------+---------+
| CV Folds           |       5 |
+--------------------+---------+

Class Distribution
+---------+------+-------------+
|   Class |   CV | Overall %   |
+=========+======+=============+
|     

## Appendix C: Classification Tuned Cross-Validation

In [ ]:
search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid={
        "n_estimators": [50, 100, 150],
        "learning_rate": [0.03, 0.1, 0.2],
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_tuned_model = search.best_estimator_

tuned_cv_report = Report(
    best_tuned_model,
    title="Appendix C: Iris Tuned Cross-Validation Report",
    author=author,
    description="Gradient Boosting tuned with GridSearchCV and evaluated with 5-fold CV.",
    theme=theme,
)

tuned_cv_report.add_crossval(X_train, y_train, cv=cv)
tuned_cv_report.add_search(search)
tuned_cv_report.build().to_pdf("reports/iris_tuned_cv_report.pdf").to_txt()

Iris Classification Report - Tuning + Cross-Validation
Lucas Summers
Gradient Boosting tuned with GridSearchCV and evaluated with 5-fold CV.

Model Overview
+-----------------+----------------------------+
| Property        | Value                      |
+=================+============================+
| Name            | GradientBoostingClassifier |
+-----------------+----------------------------+
| Type            | Classification             |
+-----------------+----------------------------+
| Sklearn         | True (v1.6.1)              |
+-----------------+----------------------------+
| Parameter Count | 20                         |
+-----------------+----------------------------+

Dataset Overview
+--------------------+---------+
| Property           |   Value |
+====================+=========+
| Features           |       4 |
+--------------------+---------+
| Total Observations |     112 |
+--------------------+---------+
| CV Folds           |       5 |
+--------------------+

## Appendix D: Classification Comparison

In [ ]:
comparison = ComparisonReport(
    reports=[
        split_report,
        cv_report,
        tuned_cv_report,
    ],
    title="Appendix D: Iris Comparison Report",
    author=author,
    description="Comparison of three classification workflows on the Iris dataset.",
    theme=theme,
)

comparison.build().to_pdf("reports/iris_comparison_report.pdf", include_model_reports=False).to_txt()

Iris Workflow Comparison
Lucas Summers
Comparison of three classification workflows on the Iris dataset.

Models

Model 1
+-------------+----------------------------------------------------------------+
| Property    | Value                                                          |
+=============+================================================================+
| Name        | GaussianNB                                                     |
+-------------+----------------------------------------------------------------+
| Description | Gaussian Naive Bayes evaluated on a standard train/test split. |
+-------------+----------------------------------------------------------------+
| Type        | Classification                                                 |
+-------------+----------------------------------------------------------------+
| Data        | Train/Test Split                                               |
+-------------+-----------------------------------------------------